# Exploratory Data Analysis — FVC2004

Analyses pixel intensity distributions, cross-sensor variation, NCC similarity separability, and class balance across train/val/test splits. All plots are saved to `artifacts/figures/`.

1. **Univariate** — pixel intensity histograms, native resolution, impressions per finger.
2. **Bivariate** — genuine vs impostor pair features, scatter plots, and a correlation matrix including the verification **label**.
3. **Class balance** — visual summaries of genuine/impostor pairs and image counts across splits, sensors, and partitions.

**Inputs:** `artifacts/manifest_with_split.csv`, `dataset_summary.json`, `pairs_*.csv`  
**Outputs:** figures under `artifacts/figures/`.

Run `01_data_engineering.ipynb` first to generate those artifacts.

In [1]:
# Imports and path setup
from __future__ import annotations

import csv
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

_cwd = Path.cwd()
if _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
FIG_DIR = ARTIFACT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("FIG_DIR:", FIG_DIR)

PROJECT_ROOT: /Users/spartan/Desktop/CS228/Project
ARTIFACT_DIR: /Users/spartan/Desktop/CS228/Project/artifacts
FIG_DIR: /Users/spartan/Desktop/CS228/Project/artifacts/figures


In [2]:
# Load manifest, pair CSVs, and dataset summary
def load_csv(path: Path) -> list[dict]:
    with path.open() as f:
        return list(csv.DictReader(f))


manifest = load_csv(ARTIFACT_DIR / "manifest_with_split.csv")
summary = json.loads((ARTIFACT_DIR / "dataset_summary.json").read_text())

print(f"Loaded {len(manifest)} image records")
print(f"Distinct finger identities: {summary['num_finger_identities']}")
print(f"Total images (summary): {summary['num_images_total']}")

Loaded 3520 image records
Distinct finger identities: 440
Total images (summary): 3520


## 1. Univariate Analysis

Per-image pixel statistics (mean, std, min, max) on grayscale normalized images. Native resolution varies by sensor — plotted as histograms and box plots grouped by sensor and subset.*).

In [3]:
# Sample images and compute per-image pixel statistics
SAMPLE_N = 2000
rng = random.Random(SEED)
sampled = rng.sample(manifest, min(SAMPLE_N, len(manifest)))

stats_list = []
for r in sampled:
    img_path = PROJECT_ROOT / r["image_path"]
    with Image.open(img_path) as img:
        w, h = img.size
        arr = np.asarray(img.convert("L"), dtype=np.float32) / 255.0
    stats_list.append(
        {
            "mean": float(arr.mean()),
            "std": float(arr.std()),
            "min": float(arr.min()),
            "max": float(arr.max()),
            "native_w": int(w),
            "native_h": int(h),
            "native_area": int(w * h),
            "partition": r["partition"],
            "sensor": r["sensor"],
            "subset": r["subset"],
            "split": r["split"],
            "finger_id": r["finger_id"],
        }
    )

means = np.array([s["mean"] for s in stats_list])
stds = np.array([s["std"] for s in stats_list])
areas = np.array([s["native_area"] for s in stats_list], dtype=np.float64)

print(f"Sampled {len(stats_list)} images")
print(f"Mean intensity: {means.mean():.4f} ± {means.std():.4f}")
print(f"Std intensity:  {stds.mean():.4f} ± {stds.std():.4f}")
print(f"Native area (pixels): min={areas.min():.0f} max={areas.max():.0f}")

Sampled 2000 images
Mean intensity: 0.6269 ± 0.2003
Std intensity:  0.2120 ± 0.0769
Native area (pixels): min=110592 max=307200


In [4]:
# Plot pixel intensity histogram and std distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(means, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Mean pixel intensity (0–1)")
axes[0].set_ylabel("Count")
axes[0].set_title("Mean intensity (sample)")

axes[1].hist(stds, bins=50, edgecolor="black", alpha=0.7, color="darkorange")
axes[1].set_xlabel("Std of pixel intensity")
axes[1].set_ylabel("Count")
axes[1].set_title("Intensity std dev (sample)")

fig.tight_layout()
fig.savefig(FIG_DIR / "hist_pixel_intensity.png")
plt.show()
print("Saved: hist_pixel_intensity.png")

Saved: hist_pixel_intensity.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_11516/2103493076.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Histogram of native image area (width × height) by sensor
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(areas, bins=40, edgecolor="black", alpha=0.75, color="slateblue")
ax.set_xlabel("Native image area (width × height pixels)")
ax.set_ylabel("Count")
ax.set_title("Native resolution area (sample) — differs by FVC2004 sensor")
fig.tight_layout()
fig.savefig(FIG_DIR / "hist_native_image_area.png")
plt.show()
print("Saved: hist_native_image_area.png")

Saved: hist_native_image_area.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_11516/976967574.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Box plot of pixel intensity grouped by sensor and subset (A vs B)
sensors = sorted(set(s["sensor"] for s in stats_list))
subsets = ["A", "B"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

data_sensor = [np.array([s["mean"] for s in stats_list if s["sensor"] == sen]) for sen in sensors]
bp1 = axes[0].boxplot(data_sensor, tick_labels=[s.replace("_", "\n") for s in sensors], patch_artist=True)
colors_s = plt.cm.Set2(np.linspace(0, 1, len(sensors)))
for patch, c in zip(bp1["boxes"], colors_s):
    patch.set_facecolor(c)
axes[0].set_ylabel("Mean pixel intensity (0–1)")
axes[0].set_title("Mean intensity by sensor (sample)")

data_sub = [np.array([s["mean"] for s in stats_list if s["subset"] == sub]) for sub in subsets]
bp2 = axes[1].boxplot(data_sub, tick_labels=[f"Subset {sub}" for sub in subsets], patch_artist=True)
for patch, c in zip(bp2["boxes"], ["#4C72B0", "#DD8452"]):
    patch.set_facecolor(c)
axes[1].set_ylabel("Mean pixel intensity (0–1)")
axes[1].set_title("Mean intensity: subset A vs B (sample)")

fig.tight_layout()
fig.savefig(FIG_DIR / "boxplot_intensity_by_group.png")
plt.show()
print("Saved: boxplot_intensity_by_group.png")

Saved: boxplot_intensity_by_group.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_11516/2206905674.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Histogram of impressions per finger identity
by_finger = defaultdict(int)
for r in manifest:
    by_finger[r["finger_id"]] += 1
impression_counts = list(by_finger.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    impression_counts,
    bins=range(min(impression_counts), max(impression_counts) + 2),
    edgecolor="black",
    alpha=0.7,
    color="mediumseagreen",
    align="left",
)
ax.set_xlabel("Impressions per finger identity")
ax.set_ylabel("Number of finger identities")
ax.set_title("Impressions per finger (protocol: 8 per identity)")
fig.tight_layout()
fig.savefig(FIG_DIR / "hist_impressions_per_finger.png")
plt.show()
print("Saved: hist_impressions_per_finger.png")

Saved: hist_impressions_per_finger.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_11516/2704122354.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Bivariate Analysis

NCC similarity and pixel intensity differences are computed for sampled pairs. Scatter plots and a correlation matrix show how these features relate to the genuine/impostor label.e verification task.

Pairs are drawn **balanced** (equal genuine / impostor) from `pairs_train.csv` so plots are not dominated by one class.

In [8]:
# Compute NCC scores for sampled genuine and impostor pairs
PAIR_PER_CLASS = 500
IMG_SIZE = (128, 128)

pairs_train = load_csv(ARTIFACT_DIR / "pairs_train.csv")
gen = [p for p in pairs_train if p["label"] == "1"]
imp = [p for p in pairs_train if p["label"] == "0"]
rng.shuffle(gen)
rng.shuffle(imp)
pair_sample = gen[:PAIR_PER_CLASS] + imp[:PAIR_PER_CLASS]
rng.shuffle(pair_sample)


def load_gray(path: Path, size=IMG_SIZE) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize(size), dtype=np.float32) / 255.0


def normalized_cross_correlation(a: np.ndarray, b: np.ndarray) -> float:
    a_flat = a.flatten()
    b_flat = b.flatten()
    a_norm = a_flat - a_flat.mean()
    b_norm = b_flat - b_flat.mean()
    denom = np.linalg.norm(a_norm) * np.linalg.norm(b_norm)
    if denom < 1e-12:
        return 0.0
    return float(np.dot(a_norm, b_norm) / denom)


sim_genuine, sim_impostor = [], []
mean_diff_genuine, mean_diff_impostor = [], []
std_diff_genuine, std_diff_impostor = [], []

for p in pair_sample:
    img1 = load_gray(PROJECT_ROOT / p["image_path_1"])
    img2 = load_gray(PROJECT_ROOT / p["image_path_2"])
    ncc = normalized_cross_correlation(img1, img2)
    mdiff = abs(float(img1.mean()) - float(img2.mean()))
    sdiff = abs(float(img1.std()) - float(img2.std()))
    label = int(p["label"])

    if label == 1:
        sim_genuine.append(ncc)
        mean_diff_genuine.append(mdiff)
        std_diff_genuine.append(sdiff)
    else:
        sim_impostor.append(ncc)
        mean_diff_impostor.append(mdiff)
        std_diff_impostor.append(sdiff)

print(f"Genuine pairs: {len(sim_genuine)}, impostor pairs: {len(sim_impostor)}")
print(f"Genuine NCC: {np.mean(sim_genuine):.4f} ± {np.std(sim_genuine):.4f}")
print(f"Impostor NCC: {np.mean(sim_impostor):.4f} ± {np.std(sim_impostor):.4f}")

Genuine pairs: 500, impostor pairs: 500
Genuine NCC: 0.3719 ± 0.2535
Impostor NCC: 0.1166 ± 0.2229


In [ ]:
# Overlapping histogram of NCC scores for genuine vs impostor pairs
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(sim_genuine, bins=40, alpha=0.6, label="Genuine", color="green", edgecolor="black", density=True)
ax.hist(sim_impostor, bins=40, alpha=0.6, label="Impostor", color="red", edgecolor="black", density=True)
ax.set_xlabel("Normalized cross-correlation (NCC)")
ax.set_ylabel("Density")
ax.set_title("NCC: genuine vs impostor (balanced train sample)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "hist_ncc_genuine_vs_impostor.png")
plt.show()
print("Saved: hist_ncc_genuine_vs_impostor.png")

In [ ]:
# Scatter plot of mean vs std intensity difference for genuine and impostor pairs
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(mean_diff_impostor, std_diff_impostor, alpha=0.35, s=14, color="red", label="Impostor")
ax.scatter(mean_diff_genuine, std_diff_genuine, alpha=0.35, s=14, color="green", label="Genuine")
ax.set_xlabel("|Mean intensity diff|")
ax.set_ylabel("|Std intensity diff|")
ax.set_title("Pairwise intensity differences vs verification label")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "scatter_mean_std_diff.png")
plt.show()
print("Saved: scatter_mean_std_diff.png")

In [ ]:
# Correlation matrix of NCC, intensity difference features, and pair label
ncc_all = sim_genuine + sim_impostor
md_all = mean_diff_genuine + mean_diff_impostor
sd_all = std_diff_genuine + std_diff_impostor
labels_vec = np.array([1] * len(sim_genuine) + [0] * len(sim_impostor))

# Rows follow same order as labels_vec
features = np.column_stack([ncc_all, md_all, sd_all, labels_vec])
feature_names = ["NCC", "|Mean Diff|", "|Std Diff|", "Label"]
corr = np.corrcoef(features.T)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha="right")
ax.set_yticklabels(feature_names)
for i in range(len(feature_names)):
    for j in range(len(feature_names)):
        ax.text(
            j,
            i,
            f"{corr[i, j]:.2f}",
            ha="center",
            va="center",
            color="white" if abs(corr[i, j]) > 0.5 else "black",
            fontsize=10,
        )
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation matrix (pair features + label)")
fig.tight_layout()
fig.savefig(FIG_DIR / "corr_matrix_pairwise_features.png")
plt.show()
print("Saved: corr_matrix_pairwise_features.png")

## 3. Class Balance

Bar charts showing genuine vs impostor pair counts and image counts across splits, sensors, and FVC partitions.

In [ ]:
# Bar chart of genuine vs impostor pair counts per split
pair_info = summary["pairs"]
splits = ["train", "val", "test"]
gen_counts = [pair_info[s]["genuine_pairs"] for s in splits]
imp_counts = [pair_info[s]["impostor_pairs"] for s in splits]

x = np.arange(len(splits))
width = 0.35
max_h = max(max(gen_counts), max(imp_counts))
pad = max(max_h * 0.02, 100)

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width / 2, gen_counts, width, label="Genuine", color="green", edgecolor="black")
bars2 = ax.bar(x + width / 2, imp_counts, width, label="Impostor", color="red", edgecolor="black")
ax.set_xlabel("Split")
ax.set_ylabel("Number of pairs")
ax.set_title("Class balance: genuine vs impostor pairs per split")
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in splits])
ax.legend()

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + pad, f"{int(h):,}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + pad, f"{int(h):,}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "class_balance_pairs.png")
plt.show()
print("Saved: class_balance_pairs.png")

In [ ]:
# Image count by partition and sensor
partitions = summary["partitions"]
part_counts = Counter(r["partition"] for r in manifest)
part_vals = [part_counts[p] for p in partitions]

sensor_order = sorted(summary["images_by_sensor"].keys(), key=lambda s: summary["images_by_sensor"][s], reverse=True)
sensor_counts = Counter(r["sensor"] for r in manifest)
sen_vals = [sensor_counts[s] for s in sensor_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(partitions, part_vals, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(partitions))), edgecolor="black")
axes[0].set_ylabel("Image count")
axes[0].set_title("Images per FVC partition (DB × subset)")
axes[0].tick_params(axis="x", rotation=45)
for i, v in enumerate(part_vals):
    axes[0].text(i, v + 15, f"{v}", ha="center", fontsize=8)

axes[1].bar(range(len(sensor_order)), sen_vals, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"], edgecolor="black")
axes[1].set_xticks(range(len(sensor_order)))
axes[1].set_xticklabels(sensor_order, rotation=55, ha="right", fontsize=8)
axes[1].set_ylabel("Image count")
axes[1].set_title("Images per sensor (880 each in full FVC2004)")
for i, v in enumerate(sen_vals):
    axes[1].text(i, v + 15, f"{v}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "image_counts_partition_sensor.png")
plt.show()
print("Saved: image_counts_partition_sensor.png")

In [ ]:
# Image count per split
split_counts = Counter(r["split"] for r in manifest)
split_order = ["train", "val", "test"]
split_vals = [split_counts[s] for s in split_order]

fig, ax = plt.subplots(figsize=(6, 4))
colors_split = ["#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar([s.capitalize() for s in split_order], split_vals, color=colors_split, edgecolor="black")
ax.set_ylabel("Image count")
ax.set_title("Images per train / val / test split")
mx = max(split_vals)
for bar, v in zip(bars, split_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + mx * 0.02, f"{v:,}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "image_counts_per_split.png")
plt.show()
print("Saved: image_counts_per_split.png")

In [ ]:
# One finger identity: all 8 impressions (native resolution preview)
sample_fid = manifest[0]["finger_id"]
sample_rows = sorted(
    [r for r in manifest if r["finger_id"] == sample_fid],
    key=lambda r: int(r["impression"]),
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes_flat = axes.flatten()
for i, r in enumerate(sample_rows[:8]):
    img = Image.open(PROJECT_ROOT / r["image_path"]).convert("L")
    axes_flat[i].imshow(np.asarray(img), cmap="gray")
    axes_flat[i].set_title(f"imp {r['impression']} — {r['partition']}", fontsize=9)
    axes_flat[i].axis("off")
for j in range(len(sample_rows), 8):
    axes_flat[j].axis("off")

fig.suptitle(f"Sample finger identity: {sample_fid} (8 impressions)", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "sample_fingerprint_grid.png")
plt.show()
print("Saved: sample_fingerprint_grid.png")

In [ ]:
# Print list of all saved figure files
saved = sorted(p.name for p in FIG_DIR.glob("*.png"))
print(f"\nFigures in {FIG_DIR} ({len(saved)} files):")
for name in saved:
    print(" -", name)